# Hands-On Machine Learning — Chapter 7
## Ensemble Learning and Random Forests


### Introduction

Ensemble methods combine multiple individual models—often called *base learners*—to create a stronger and more robust predictive model. The core idea is that a group of weak learners, when aggregated, can produce better generalization than any single model alone.

This chapter introduces key ensemble techniques such as **bagging**, **boosting**, and **stacking**, with a detailed focus on **Random Forests**. These methods form the foundation of many state-of-the-art models used in practice.

<p align="left"><img src="../fig/figure7.1.png" width="45%"></p>

### Why Ensemble Learning Works

Ensemble learning exploits the concept of **diversity among models**. If individual models make uncorrelated errors, their combined prediction averages out the noise, reducing overall variance.

Consider $M$ independent models with errors $\epsilon_i$. The variance of the averaged ensemble prediction is:

$$ Var(\bar{y}) = \frac{1}{M^2} \sum Var(\epsilon_i) = \frac{\sigma^2}{M} $$

Thus, more independent models → lower variance. Diversity can be achieved by training models on **different subsets of data, features, or initializations**.

<p align="left"><img src="../fig/figure7.2.png" width="45%"></p>

### Voting Classifiers

The simplest ensemble method is **majority voting** (for classification) or **averaging** (for regression). If each base classifier outputs a class prediction, the ensemble predicts the majority class.

- **Hard voting:** based on class labels.
- **Soft voting:** based on predicted probabilities, giving higher weight to confident models.

Soft voting generally performs better if the base models are well-calibrated.

<p align="left"><img src="../fig/figure7.3.png" width="45%"></p>

In [ ]:
# Example: Voting Classifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

log_clf = LogisticRegression()
svc_clf = SVC(probability=True)
tree_clf = DecisionTreeClassifier(max_depth=5)

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('svc', svc_clf), ('tree', tree_clf)],
    voting='soft'
)

voting_clf.fit(X_train, y_train)
for clf in (log_clf, svc_clf, tree_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

### Bagging and Pasting

**Bagging** (Bootstrap Aggregating) and **Pasting** train the same model type on different random subsets of the training data.

- **Bagging:** samples with replacement (bootstrap sampling).
- **Pasting:** samples without replacement.

Each model is trained independently, and predictions are averaged (regression) or voted (classification). Bagging reduces variance while keeping bias roughly the same.

Mathematically, ensemble prediction is:

$$ \hat{y}(x) = \frac{1}{M} \sum_{m=1}^M \hat{y}_m(x) $$

<p align="left"><img src="../fig/figure7.4.png" width="45%"></p>

In [ ]:
# Bagging Example with Decision Trees
from sklearn.ensemble import BaggingClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=100,
    max_samples=100, bootstrap=True, random_state=42
)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)
print('Bagging accuracy:', accuracy_score(y_test, y_pred))

### Out-of-Bag Evaluation

Because bagging uses sampling with replacement, some instances are not included in each bootstrap sample—these are called **out-of-bag (OOB)** samples.

We can evaluate each model on its OOB samples to estimate performance **without a separate validation set**.

Scikit-Learn can compute OOB scores automatically via `oob_score=True`.

<p align="left"><img src="../fig/figure7.5.png" width="45%"></p>

In [ ]:
bag_clf_oob = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=200,
    bootstrap=True, oob_score=True, random_state=42
)
bag_clf_oob.fit(X_train, y_train)
print('OOB score:', bag_clf_oob.oob_score_)

### Random Forests

A **Random Forest** is an ensemble of Decision Trees trained via bagging but with an extra layer of randomness:

- Each tree is trained on a bootstrap sample.
- At each split, a random subset of features is considered.

This decorrelates trees, reducing variance further. Random Forests are robust, fast, and among the most widely used algorithms in practice.

<p align="left"><img src="../fig/figure7.6.png" width="45%"></p>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200, max_leaf_nodes=16, n_jobs=-1, random_state=42
)
rf_clf.fit(X_train, y_train)
print('Random Forest accuracy:', accuracy_score(y_test, rf_clf.predict(X_test)))

### Feature Importance in Random Forests

Random Forests provide feature importance by measuring the total reduction in impurity contributed by each feature across all trees.

This makes them powerful for both prediction and interpretability.

<p align="left"><img src="../fig/figure7.7.png" width="45%"></p>

### Boosting

Boosting builds an ensemble sequentially: each new model corrects the errors made by the previous ones. It focuses on difficult cases, giving them more weight during training.

Two major boosting techniques are:
- **AdaBoost (Adaptive Boosting)**
- **Gradient Boosting**

<p align="left"><img src="../fig/figure7.8.png" width="45%"></p>

### AdaBoost (Adaptive Boosting)

In AdaBoost, each instance’s weight $w_i$ increases if it is misclassified. The next model focuses more on those hard examples.

Each weak learner is trained sequentially, and their predictions are combined via weighted voting based on learner accuracy.

Mathematically:

$$ \alpha_m = \eta \log \frac{1 - err_m}{err_m} $$

$$ F(x) = \text{sign}\left(\sum_m \alpha_m h_m(x)\right) $$

<p align="left"><img src="../fig/figure7.9.png" width="45%"></p>

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200, learning_rate=0.5, random_state=42
)
ada_clf.fit(X_train, y_train)
print('AdaBoost accuracy:', accuracy_score(y_test, ada_clf.predict(X_test)))

### Gradient Boosting

Gradient Boosting takes a different approach: it fits each new model to the **residual errors** of the combined ensemble so far.

It uses the gradient of the loss function to iteratively minimize prediction errors.

For squared error loss:

$$ r_i = y_i - F_{m-1}(x_i) $$

A new model $h_m(x)$ is trained to predict these residuals. The ensemble is updated as:

$$ F_m(x) = F_{m-1}(x) + \eta h_m(x) $$

where $\eta$ is the learning rate.

<p align="left"><img src="../fig/figure7.10.png" width="45%"></p>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train, y_train)
print('Gradient Boosting accuracy:', accuracy_score(y_test, gb_clf.predict(X_test)))

### Stacking

**Stacking (stacked generalization)** combines multiple base learners using a **meta-learner** that learns how to best combine their outputs.

Steps:
1. Train several base models on the training data.
2. Collect their predictions on a validation set.
3. Use these predictions as input features to train the meta-model.

This allows the meta-learner to correct systematic errors from base models.

<p align="left"><img src="../fig/figure7.11.png" width="45%"></p>

### Summary and Best Practices

- **Voting and averaging** aggregate predictions from diverse models.
- **Bagging** and **pasting** reduce variance by training models on random subsets.
- **Random Forests** add feature randomness for further decorrelation.
- **Boosting** (AdaBoost, Gradient Boosting) reduces bias by focusing on difficult cases.
- **Stacking** learns how to optimally combine different model types.
- Ensemble methods often outperform single models but increase computational cost.